<a href="https://colab.research.google.com/github/lathikasuriyamoorthy-29/Churn_Prediction_lathikasuriyamoorthy/blob/main/cleaning_%26_model_testing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving churnguard_data.csv to churnguard_data (4).csv


In [ ]:
import pandas as pd


In [ ]:
df = pd.read_csv("churnguard_data.csv")

print("Dataset loaded successfully!")
print(df.shape)

Dataset loaded successfully!
(1030, 12)


In [ ]:
df = df.drop(columns = ["customerID"])
df = df.drop_duplicates()
df["gender"] = df["gender"].str.strip()
df["PaymentMethod"] = df["PaymentMethod"].str.strip()
df["Churn"] = df["Churn"].str.strip().str.title()
df["PhoneService"] = df["PhoneService"].str.strip().str.title()
df["PaperlessBilling"] = df["PaperlessBilling"].str.strip().str.title()

In [ ]:
original_df = pd.read_csv("churnguard_data.csv")

print("Original missing Contract values:",
      original_df["Contract"].isna().sum())

print("Original missing InternetService values:",
      original_df["InternetService"].isna().sum())

Original missing Contract values: 0
Original missing InternetService values: 16


In [ ]:
print(df["Contract"].value_counts())

Contract
Month-to-month    466
One year          187
Two year          158
month-to-month     42
month to month     41
Monthly            35
One Year           17
1 year             13
one year           12
2 year             11
Two Year            9
two year            9
Name: count, dtype: int64


In [ ]:
print(df["InternetService"].value_counts())

InternetService
Fiber optic    336
DSL            317
No             185
Fibre optic     31
fiber optic     24
dsl             21
DSl             19
FiberOptic      17
Dsl             15
NO              13
no               6
Name: count, dtype: int64


In [ ]:
print(df["Contract"].unique())

['Month-to-month' 'Two year' 'month to month' 'One year' 'month-to-month'
 'Monthly' 'Two Year' '1 year' '2 year' 'two year' 'One Year' 'one year']


In [ ]:
print(df["Contract"].unique())

['Month-to-month' 'Two year' 'month to month' 'One year' 'month-to-month'
 'Monthly' 'Two Year' '1 year' '2 year' 'two year' 'One Year' 'one year']


In [ ]:
df["Contract"] = df["Contract"].str.strip().str.lower()

df["Contract"] = df["Contract"].replace({
    "month-to-month": "Month-to-month",
    "month to month": "Month-to-month",
    "monthly": "Month-to-month",

    "one year": "One year",
    "1 year": "One year",

    "two year": "Two year",
    "2 year": "Two year"
})

In [ ]:
print(df["Contract"].unique())

['Month-to-month' 'Two year' 'One year']


In [ ]:
df["Contract"].isnull().sum()

np.int64(0)

In [ ]:
df["InternetService"] = df["InternetService"].str.strip().str.lower()

df["InternetService"] = df["InternetService"].replace({
    "fiber optic": "Fiber optic",
    "fibre optic": "Fiber optic",
    "fiberoptic": "Fiber optic",

    "dsl": "DSL",

    "no": "No"
})

In [ ]:
print(df["InternetService"].unique())

['Fiber optic' 'DSL' 'No' nan]


In [ ]:
df["InternetService"].isnull().sum()

np.int64(16)

In [ ]:
print(df["InternetService"].value_counts(dropna=False))

InternetService
Fiber optic    408
DSL            372
No             204
NaN             16
Name: count, dtype: int64


In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"],errors = "coerce")
df = df[df["tenure"]>0]
df = df[(df["MonthlyCharges"] >= 10)& (df["MonthlyCharges"]<= 200)]
df["MonthlyCharges"] = df["MonthlyCharges"].fillna(df["MonthlyCharges"].mean())
df["TotalCharges"] = df["TotalCharges"].fillna(df["TotalCharges"].mean())
tenure_median = round(df["tenure"].median())
df["tenure"] = df["tenure"].fillna(tenure_median).astype(int)
print("Shape of cleaned DataFrame:", df.shape)
print("\nmissing values:")
print(df.isnull().sum())

Shape of cleaned DataFrame: (867, 11)

missing values:
gender               0
SeniorCitizen        0
tenure               0
PhoneService         0
InternetService     14
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges         0
Churn                0
dtype: int64


In [ ]:
df.to_csv("churnguard_cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [ ]:
df["InternetService"] = df["InternetService"].fillna(
    df["InternetService"].mode()[0]
)

In [ ]:
print(df.isna().sum())

gender              0
SeniorCitizen       0
tenure              0
PhoneService        0
InternetService     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64


In [ ]:
df["Churn"] = df["Churn"].map({
    "Yes": 1,
    "No": 0
})

In [ ]:
print(df["Churn"].value_counts())

Churn
0    593
1    274
Name: count, dtype: int64


In [ ]:
categorical_columns = [
    "gender",
    "PhoneService",
    "InternetService",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True
)

In [ ]:
y = df["Churn"]

In [ ]:
X = df.drop(columns=["Churn"])

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
model = LogisticRegression(max_iter=1000)

In [ ]:
model.fit(X_train, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

Accuracy: 0.7011494252873564


In [ ]:
print("\nClassification Report:")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=["Stay", "Churn"]
    )
)


Classification Report:
              precision    recall  f1-score   support

        Stay       0.75      0.85      0.80       122
       Churn       0.50      0.35      0.41        52

    accuracy                           0.70       174
   macro avg       0.63      0.60      0.60       174
weighted avg       0.68      0.70      0.68       174

